In [1]:
# =========================================
# ORDERS + ORDER ITEMS – INSPECTION & BUSY ANALYSIS
# =========================================

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# -----------------------------
# 1. Load data
# -----------------------------
orders = pd.read_csv("../datajackies/ordersjackies.csv")
order_items = pd.read_csv("../datajackies/orderitemsjackies.csv")

# -----------------------------
# 2. Preview
# -----------------------------
print("=== ORDERS PREVIEW ===")
display(orders.head())

print("\n=== ORDER ITEMS PREVIEW ===")
display(order_items.head())

# -----------------------------
# 3. Shape
# -----------------------------
print("\n=== DATA SHAPES ===")
print("Orders:", orders.shape)
print("Order Items:", order_items.shape)

# -----------------------------
# 4. Data types
# -----------------------------
print("\n=== DATA TYPES ===")
print("\nORDERS:")
print(orders.dtypes)

print("\nORDER ITEMS:")
print(order_items.dtypes)

# -----------------------------
# 5. Missing values
# -----------------------------
print("\n=== MISSING VALUES ===")
print("\nORDERS:")
print(orders.isnull().sum())

print("\nORDER ITEMS:")
print(order_items.isnull().sum())

# -------------------------------------------------
# 6. BUSYNESS ANALYSIS (VISUAL)
# -------------------------------------------------

# Convert order_time to datetime and extract hour
orders["order_time"] = pd.to_datetime(orders["order_time"], errors="coerce")
orders["hour"] = orders["order_time"].dt.hour

# Helper: 24h -> AM/PM
def hour_to_label(h):
    if h == 0:
        return "12 AM"
    elif h < 12:
        return f"{h} AM"
    elif h == 12:
        return "12 PM"
    else:
        return f"{h-12} PM"

# -----------------------------
# Plot 1: Weekday vs Weekend busyness
# -----------------------------
plt.figure(figsize=(6, 4))
sns.countplot(data=orders, x="day_type")
plt.title("Order Volume: Weekday vs Weekend")
plt.xlabel("Day Type")
plt.ylabel("Number of Orders")
plt.tight_layout()
plt.show()

# -----------------------------
# Plot 2: Busyness by Time (Normal Time)
# -----------------------------
hour_counts = orders["hour"].value_counts().sort_index()
hour_labels = [hour_to_label(h) for h in hour_counts.index]

plt.figure(figsize=(12, 4))
plt.bar(hour_labels, hour_counts.values)
plt.title("Order Volume by Time of Day")
plt.xlabel("Time")
plt.ylabel("Number of Orders")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# -------------------------------------------------
# 7. GROUP BEHAVIOR ANALYSIS (JUSTIFICATION)
# -------------------------------------------------

# Total quantity per order
order_size = (
    order_items
    .groupby("order_id")["quantity"]
    .sum()
    .reset_index(name="total_items")
)

# Merge with day_type
order_size = order_size.merge(
    orders[["order_id", "day_type"]],
    on="order_id",
    how="left"
)

# Define group type
order_size["group_type"] = order_size["total_items"].apply(
    lambda x: "Group (>2 items)" if x > 2 else "Alone (≤2 items)"
)

# Plot: Group vs Alone by day type
plt.figure(figsize=(8, 4))
sns.countplot(
    data=order_size,
    x="day_type",
    hue="group_type"
)
plt.title("Group vs Alone Orders by Day Type")
plt.xlabel("Day Type")
plt.ylabel("Number of Orders")
plt.tight_layout()
plt.show()


ModuleNotFoundError: No module named 'seaborn'

In [33]:
# ==================================================
# FULL END-TO-END BUSY LEVEL PREDICTOR (ONE CELL)
# ==================================================

import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report


# -----------------------------
# 2. Basic cleaning
# -----------------------------
orders = orders.drop_duplicates()
order_items = order_items.drop_duplicates()

orders = orders.dropna(subset=["order_id", "order_time", "day_type"])
order_items = order_items.dropna(subset=["order_id", "quantity"])
order_items["quantity"] = order_items["quantity"].astype(int)

# -----------------------------
# 3. Create hour from order_time
# -----------------------------
orders["order_time"] = pd.to_datetime(
    orders["order_time"], format="%H:%M", errors="coerce"
)
orders["hour"] = orders["order_time"].dt.hour
orders = orders.dropna(subset=["hour"])

# -----------------------------
# 4. Group behavior per order
# -----------------------------
order_size = (
    order_items
    .groupby("order_id")["quantity"]
    .sum()
    .reset_index(name="total_items")
)

order_size["group_type"] = order_size["total_items"].apply(
    lambda x: "alone" if x <= 2 else "group"
)

# -----------------------------
# 5. Merge order-level features
# -----------------------------
df = orders.merge(
    order_size[["order_id", "group_type"]],
    on="order_id",
    how="inner"
)

# Military time (single time feature)
df["time"] = df["hour"].apply(lambda h: f"{int(h):02d}:00")

df = df[["order_id", "time", "day_type", "group_type"]]

# -----------------------------
# 6. Create busy_level PER ORDER
# -----------------------------
hour_load = (
    df.groupby("time")
      .size()
      .reset_index(name="orders_per_hour")
)

df = df.merge(hour_load, on="time", how="left")

low_thr = hour_load["orders_per_hour"].quantile(0.33)
high_thr = hour_load["orders_per_hour"].quantile(0.66)

def label_busy(x):
    if x <= low_thr:
        return "Low"
    elif x <= high_thr:
        return "Medium"
    else:
        return "High"

df["busy_level"] = df["orders_per_hour"].apply(label_busy)

# -----------------------------
# 7. FINAL DATAFRAME
# -----------------------------
final_df = df[["time", "day_type", "group_type", "busy_level"]]

print("=== FINAL DATAFRAME USED FOR TRAINING ===")
display(final_df.head())
print("Final shape:", final_df.shape)

# -----------------------------
# 8. Train / test split
# -----------------------------
X = final_df[["time", "day_type", "group_type"]]
y = final_df["busy_level"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# -----------------------------
# 9. Pipeline (encoding + model)
# -----------------------------
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"),
         ["time", "day_type", "group_type"])
    ]
)

model = RandomForestClassifier(
    n_estimators=300,
    random_state=42
)

pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", model)
])

# -----------------------------
# 10. Train & evaluate
# -----------------------------
pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)

print("\nAccuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

# -----------------------------
# 11. Save pipeline
# -----------------------------
joblib.dump(pipeline, "busy_level_pipeline.joblib")
print("\nSaved model: busy_level_pipeline.joblib")


=== FINAL DATAFRAME USED FOR TRAINING ===


,time,day_type,group_type,busy_level
0,22:00,weekday,group,High
1,16:00,weekday,alone,Medium
2,12:00,weekday,alone,Medium
3,16:00,weekday,alone,Medium
4,22:00,weekday,alone,High


Final shape: (1917, 4)

Accuracy: 1.0

Classification Report:

              precision    recall  f1-score   support

        High       1.00      1.00      1.00       270
         Low       1.00      1.00      1.00        52
      Medium       1.00      1.00      1.00        62

    accuracy                           1.00       384
   macro avg       1.00      1.00      1.00       384
weighted avg       1.00      1.00      1.00       384


Saved model: busy_level_pipeline.joblib
